# VAJRA — Kaggle Persistent Worker

One-cell runtime appliance for the VAJRA Qwen worker. It installs Ollama only when missing, reuses the existing model store, starts the VAJRA HTTP worker, performs a real inference smoke test, then keeps the supervisor in the foreground so Kaggle does not reap its child processes when the cell exits.

**Runtime persistence:** this keeps the worker alive for the lifetime of the Kaggle runtime/cell. Kaggle may still destroy or reset a free runtime; that cannot be made permanently persistent by notebook code.

In [ ]:
%%bash
set -euo pipefail

REPO=/kaggle/working/vajra
MODEL=qwen2.5-coder:32b
MODEL_STORE=/root/.ollama/models

echo '=== VAJRA KAGGLE WORKER — ONE CELL ==='

if [ ! -d "$REPO/.git" ]; then
  echo 'Cloning VAJRA...'
  git clone -q https://github.com/Exploiter69/vajra.git "$REPO"
else
  echo 'Refreshing VAJRA checkout...'
  git -C "$REPO" fetch -q origin main
  git -C "$REPO" reset --hard -q origin/main
fi

cd "$REPO"
export PYTHONPATH="$REPO/src"
export OLLAMA_MODELS="$MODEL_STORE"
export OLLAMA_HOST='127.0.0.1:11434'
export VAJRA_WORKER_HOST='127.0.0.1'
export VAJRA_WORKER_PORT='8787'
export VAJRA_WORKER_MODEL="$MODEL"
export VAJRA_WORKER_ID='kaggle-t4-persistent-01'

echo "Revision: $(git rev-parse --short HEAD)"
echo "Model store: $(du -sh "$MODEL_STORE" 2>/dev/null || echo missing)"

if [ -x /kaggle/working/ollama/bin/ollama ]; then
  export VAJRA_OLLAMA_BIN=/kaggle/working/ollama/bin/ollama
elif command -v ollama >/dev/null 2>&1; then
  export VAJRA_OLLAMA_BIN="$(command -v ollama)"
else
  echo 'Installing Ollama once...'
  curl -fsSL https://ollama.com/install.sh | sh
  if [ -x /usr/local/bin/ollama ]; then
    export VAJRA_OLLAMA_BIN=/usr/local/bin/ollama
  else
    export VAJRA_OLLAMA_BIN="$(command -v ollama)"
  fi
fi

echo "Ollama: $VAJRA_OLLAMA_BIN"
"$VAJRA_OLLAMA_BIN" --version

echo 'Starting VAJRA supervisor...'
python scripts/start_kaggle_http_worker.py > /kaggle/working/vajra-worker.log 2>&1 &
SUPERVISOR_PID=$!

cleanup() {
  kill "$SUPERVISOR_PID" 2>/dev/null || true
  wait "$SUPERVISOR_PID" 2>/dev/null || true
}
trap cleanup EXIT INT TERM

echo "Supervisor PID: $SUPERVISOR_PID"
echo 'Waiting for worker readiness...'

READY=0
for i in $(seq 1 180); do
  if curl -fsS --max-time 3 http://127.0.0.1:8787/health > /tmp/vajra-health.json 2>/dev/null; then
    READY=1
    cat /tmp/vajra-health.json
    break
  fi
  if ! kill -0 "$SUPERVISOR_PID" 2>/dev/null; then
    echo 'Supervisor exited unexpectedly:'
    cat /kaggle/working/vajra-worker.log || true
    exit 1
  fi
  sleep 1
done

if [ "$READY" -ne 1 ]; then
  echo 'Worker did not become ready.'
  tail -100 /kaggle/working/vajra-worker.log || true
  exit 1
fi

echo '=== REAL INFERENCE SMOKE TEST ==='
python - <<'PY'
import json, subprocess, urllib.request, uuid

from vajra.runtime.worker_protocol import WorkerJob
from vajra.runtime.kaggle_worker import KaggleWorkerAdapter

job = WorkerJob(
    run_id='kaggle-persistent-smoke',
    step_id='smoke-step',
    attempt_id='smoke-attempt',
    repository_revision=subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(),
    workspace_contract={'mode':'read_only_smoke'},
    context_bundle={'prompt':'Return exactly VAJRA_PERSISTENT_WORKER_OK'},
    allowed_capabilities=('completion',),
    budget={'max_output_tokens':32,'timeout_seconds':180},
    deadline='2099-01-01T00:00:00Z',
    expected_output_schema={'type':'string'},
    correlation_id=str(uuid.uuid4()),
)
adapter = KaggleWorkerAdapter()
body = adapter.encode_job(job).encode()
req = urllib.request.Request(
    'http://127.0.0.1:8787/infer',
    data=body,
    headers={'Content-Type':'application/json'},
    method='POST',
)
with urllib.request.urlopen(req, timeout=190) as response:
    result = json.loads(response.read().decode())
print(json.dumps(result, indent=2))
PY

echo '=== VAJRA WORKER READY ==='
echo 'The supervisor is intentionally kept in the foreground.'
echo 'Leave this cell running while VAJRA uses the worker.'
echo 'Model pull will be skipped whenever the model is already present.'

wait "$SUPERVISOR_PID"
